In [0]:
#Leitura das bases da camada silver
df_associado = spark.read.table("workspace.silver.associado_diaria_05hs")
df_conta = spark.read.table("workspace.silver.conta_diaria_05hs")
df_cartao = spark.read.table("workspace.silver.cartao_diaria_05hs")
df_movimento = spark.read.table("workspace.silver.movimento_diaria_05hs")


In [0]:
# Transforma para o modelo flat a tabela final.
import pyspark.sql.functions as F

movimento_flat = (
    df_movimento.alias("m")
    .join(df_cartao.alias("c"), F.col("m.id_cartao") == F.col("c.id"), "left")
    .join(df_conta.alias("co"), F.col("c.id_conta") == F.col("co.id"), "left")
    .join(df_associado.alias("a"), F.col("co.id_associado") == F.col("a.id"), "left")
    .select(
        F.initcap(F.trim(F.col("a.nome"))).cast("string").alias("nome_associado"),
        F.initcap(F.trim(F.col("a.sobrenome"))).cast("string").alias("sobrenome_associado"),
        F.col("a.idade").cast("int").alias("idade_associado"),
        F.col("m.vlr_transacao").cast("decimal(18,2)").alias("vlr_transacao_movimento"),
        F.initcap(F.trim(F.col("m.des_transacao"))).cast("string").alias("des_transacao_movimento"),
        F.date_format(F.col("m.data_movimento"), "dd/MM/yyyy").alias("data_movimento"),
        F.concat(
            F.substring(F.col("c.num_cartao").cast("string"), 1, 4),
            F.lit("********"),
            F.expr("right(cast(c.num_cartao as string), 4)"),
        ).cast("string").alias("numero_cartao"),
        F.initcap(F.trim(F.col("c.nom_impresso"))).cast("string").alias("nome_impresso_cartao"),
        F.date_format(F.col("c.data_criacao_cartao"), "dd/MM/yyyy").alias("data_criacao_cartao"),
        F.initcap(F.trim(F.col("co.tipo"))).cast("string").alias("tipo_conta"),
        F.date_format(F.col("co.data_criacao_conta"), "dd/MM/yyyy").alias("data_criacao_conta"),
    )
)

#display(movimento_flat)

In [0]:
# Leitura da última base disponível na camada gold
df_gold_prev = spark.read.table("workspace.gold.movimento_flat_diaria_06hs")

# Conta de registros da base atual e da última disponível
count_atual = movimento_flat.count()
count_prev = df_gold_prev.count()

# Define o percentual mínimo aceitável para passar (exemplo: 90%)
percentual_minimo = 90.0

# Calcula o índice de qualidade em percentual
indice_qualidade = 0 if count_prev == 0 else round((count_atual / count_prev) * 100, 2)
valida_passou = indice_qualidade >= percentual_minimo

# Exibe as quantidades, o índice de qualidade e o percentual mínimo para comparação
display(
    spark.createDataFrame(
        [(count_prev, count_atual, indice_qualidade, percentual_minimo, valida_passou)],
        ["quantidade_gold_anterior", "quantidade_atual", "indice_qualidade_percentual", "percentual_minimo", "passou"]
    )
)

In [0]:
#Salvando base na camada gold
(movimento_flat.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("gold.movimento_flat_diaria_06hs"))
